# 🎙️ AI-Based Dysarthric Speech Recognition System

### Using Data Augmentation and Pretrained ASR Models

**Course:** Natural Language Processing & Speech Processing
**Dataset:** TORGO (Dysarthric Speech) / Common Voice / Lightweight Speech Samples
**Models:** Whisper (fine-tuned with LoRA) + wav2vec2 (baseline)
**NLP Layer:** Grammar Correction (T5) | Intent Classification (BART) | Named Entity Recognition (spaCy) | Sentiment Analysis (DistilBERT)

---

# 📌 Project Overview

Dysarthric speech recognition is a challenging task due to slurred pronunciation, reduced articulation clarity, and inconsistent speech patterns. Traditional Automatic Speech Recognition (ASR) systems often struggle to accurately transcribe such speech.

This project presents a lightweight and deployable end-to-end dysarthric speech recognition pipeline using pretrained ASR models and downstream Natural Language Processing (NLP) components. The system integrates speech preprocessing, transcription, grammar correction, intent detection, named entity recognition, sentiment analysis, evaluation metrics, and an interactive Gradio-based deployment interface.

---

# 🗺️ Pipeline Overview

Environment Setup → Dataset Loading → Audio Preprocessing & Augmentation → Baseline ASR (wav2vec2) → Whisper LoRA Fine-Tuning → NLP Processing → Evaluation Metrics → Interactive Gradio Demo

---

# 📚 Research Gaps Addressed

| Gap                 | Existing Limitation                                                  | Proposed Solution                                                 |
| ------------------- | -------------------------------------------------------------------- | ----------------------------------------------------------------- |
| Implementation Gap  | Research papers focus mainly on methodology without complete systems | Developed a complete end-to-end working pipeline                  |
| Deployment Gap      | Lack of user-facing applications                                     | Integrated an interactive Gradio-based web interface              |
| Feasibility Gap     | High computational requirements for training                         | Used lightweight pretrained ASR models with LoRA fine-tuning      |
| Accessibility Gap   | Limited focus on usability for real users                            | Added simple microphone/upload-based interaction                  |
| Reproducibility Gap | Difficult to reproduce complete pipelines                            | Implemented the entire workflow in a single reproducible notebook |

---

# 🎯 Objectives

* Build an end-to-end dysarthric speech recognition pipeline
* Compare baseline and fine-tuned ASR performance
* Integrate downstream NLP processing for structured understanding
* Apply lightweight audio augmentation techniques
* Evaluate transcription quality using WER and CER metrics
* Provide an interactive and deployable web-based interface

---

# ⚙️ Notebook Workflow

The notebook is organized into the following stages:

| Cell   | Description                                       |
| ------ | ------------------------------------------------- |
| Cell 1 | Dependency installation and environment setup     |
| Cell 2 | Environment verification and GPU detection        |
| Cell 3 | Dataset loading and audio augmentation            |
| Cell 4 | Baseline ASR using wav2vec2                       |
| Cell 5 | Whisper fine-tuning using LoRA                    |
| Cell 6 | Loading NLP models                                |
| Cell 7 | NLP processing pipeline                           |
| Cell 8 | Evaluation using WER, CER, F1-score, and accuracy |
| Cell 9 | Interactive Gradio deployment                     |

---

# 🧠 Technologies Used

* Python
* PyTorch
* Hugging Face Transformers
* PEFT (LoRA)
* librosa
* audiomentations
* spaCy
* scikit-learn
* Gradio

---

# 📊 Evaluation Metrics

The project evaluates ASR and NLP performance using:

* Word Error Rate (WER)
* Character Error Rate (CER)
* Sentiment Classification Accuracy
* Macro F1-score

---

# 🚀 Expected Outcome

The final system provides:

* Speech-to-text transcription
* Grammar correction
* Intent recognition
* Named entity extraction
* Sentiment analysis
* Real-time interactive deployment through Gradio

The project demonstrates the feasibility of building an accessible and lightweight dysarthric speech recognition system using pretrained transformer-based architectures and parameter-efficient fine-tuning methods.

## ⚙️ CELL 1 — Dependency Installation and Environment Setup

This section installs all required libraries and pinned dependency versions needed for the project. Compatible package versions are used to avoid conflicts between transformers, tokenizers, PEFT, and Hugging Face components.

After installation, the runtime session must be restarted before continuing.

In [ ]:
# ================================================================
# CELL 1 — Clean install (run ONCE, then Runtime → Restart session)
# ================================================================

import subprocess, sys

print("Uninstalling conflicting packages...")
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y",
    "transformers", "huggingface_hub", "tokenizers", "peft",
    "accelerate", "torchao"
])

print("Installing pinned compatible versions...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    # Core
    "transformers==4.40.2",
    "huggingface_hub==0.23.4",
    "tokenizers==0.19.1",
    # Training
    "peft==0.11.1",
    "accelerate==0.30.1",
    # Audio
    "librosa",
    "soundfile",
    "audiomentations",
    "datasets",
    # NLP / eval
    "spacy",
    "jiwer",
    "scikit-learn",
    "sentencepiece",
    "pandas",
    # Gradio — pinned to avoid gradio_client schema bug
    "gradio==3.50.2",
    "gradio_client==0.6.1",
])

print("Downloading spaCy model...")
subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])

print("=" * 55)
print("✓ Done.")
print("➡  NOW: Runtime → Restart session")
print("   Then run Cell 2 onwards IN ORDER.")
print("=" * 55)

Uninstalling conflicting packages...
Installing pinned compatible versions...
✓ Done.
➡  NOW: Runtime → Restart session
   Then run Cell 2 onwards IN ORDER.


## ✅ CELL 2 — Environment Verification

This section verifies the installed package versions and checks GPU availability. The runtime device configuration is initialized for either CUDA or CPU execution.

In [ ]:
# ================================================================
# CELL 2 — Verify environment (run after restart)
# ================================================================

import torch
import transformers
import huggingface_hub

print(f"transformers    : {transformers.__version__}")   # must be 4.40.2
print(f"huggingface_hub : {huggingface_hub.__version__}") # must be 0.23.4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device          : {DEVICE}")

assert transformers.__version__.startswith("4.40"), \
    "Wrong transformers version! Run Cell 1 again then restart."

print("\n✓ Environment OK — continue to Cell 3.")

transformers    : 4.40.2
huggingface_hub : 0.23.4
Device          : cuda

✓ Environment OK — continue to Cell 3.


## 📂 CELL 3 — Dataset Loading and Audio Augmentation

This section loads speech samples and applies audio augmentation techniques such as Gaussian noise addition, pitch shifting, and time stretching. These augmentations improve robustness and increase variability in training data for ASR models.

In [ ]:
# ================================================================
# CELL 3 — Load dataset + audio augmentation
# ================================================================

import os
import numpy as np
import soundfile as sf
import warnings
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
from datasets import load_dataset, Audio

warnings.filterwarnings("ignore")

print("Loading dataset...")
dataset = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy",
    "clean",
    split="validation"
)
dataset = dataset.cast_column("audio", Audio())
print(f"✓ Dataset loaded: {len(dataset)} samples")

augment = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.005, p=0.5),
    TimeStretch(min_rate=0.85, max_rate=1.15, p=0.5),
    PitchShift(min_semitones=-2, max_semitones=2, p=0.5),
])

os.makedirs("augmented_audio", exist_ok=True)
os.makedirs("original_audio",  exist_ok=True)

subset = dataset.select(range(min(10, len(dataset))))
augmented_samples = []

for i in range(len(subset)):
    try:
        sample      = subset[i]
        audio_array = np.array(sample["audio"]["array"], dtype=np.float32)
        sr          = sample["audio"]["sampling_rate"]

        orig_path = f"original_audio/sample_{i}.wav"
        sf.write(orig_path, audio_array, sr)

        augmented = augment(samples=audio_array, sample_rate=sr)
        aug_path  = f"augmented_audio/sample_{i}_aug.wav"
        sf.write(aug_path, augmented, sr)

        augmented_samples.append({
            "original_path":   orig_path,
            "augmented_path":  aug_path,
            "text":            sample.get("text", ""),
            "sr":              sr
        })
        print(f"  ✓ Sample {i} processed")

    except Exception as e:
        print(f"  ✗ Sample {i} skipped: {e}")

print(f"\n✓ Augmented {len(augmented_samples)} samples ready.")

Loading dataset...
✓ Dataset loaded: 73 samples
  ✓ Sample 0 processed
  ✓ Sample 1 processed
  ✓ Sample 2 processed
  ✓ Sample 3 processed
  ✓ Sample 4 processed
  ✓ Sample 5 processed
  ✓ Sample 6 processed
  ✓ Sample 7 processed
  ✓ Sample 8 processed
  ✓ Sample 9 processed

✓ Augmented 10 samples ready.


## 🎤 CELL 4 — Baseline ASR using wav2vec2

This section loads the pretrained wav2vec2 model and performs baseline speech transcription. The model converts speech audio into raw text transcripts for comparison with the fine-tuned Whisper model.

In [ ]:
# ================================================================
# CELL 4 — wav2vec2 baseline ASR
# ================================================================

import torch
import librosa
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

print("Loading wav2vec2...")
w2v_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
w2v_model     = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h").to(DEVICE)
w2v_model.eval()
print("✓ wav2vec2 loaded.")

def transcribe_wav2vec2(audio_path):
    audio, _  = librosa.load(audio_path, sr=16000)
    inputs    = w2v_processor(
        audio, sampling_rate=16000, return_tensors="pt", padding=True
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        logits = w2v_model(**inputs).logits
    ids = torch.argmax(logits, dim=-1)
    return w2v_processor.batch_decode(ids)[0].lower()

# Quick test
if augmented_samples:
    pred = transcribe_wav2vec2(augmented_samples[0]["original_path"])
    ref  = augmented_samples[0]["text"].lower()
    print(f"\nTest:")
    print(f"  Reference : {ref}")
    print(f"  Predicted : {pred}")
    print("✓ wav2vec2 working.")
else:
    print("✗ No samples — run Cell 3 first.")

Loading wav2vec2...


Some weights of the model checkpoint at facebook/wav2vec2-base-960h were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1', 'wav2vec2.masked_spec_embed']
You sho

✓ wav2vec2 loaded.

Test:
  Reference : mister quilter is the apostle of the middle classes and we are glad to welcome his gospel
  Predicted : mister quilter is the apostle of the middle classes and we are glad to welcome his gospel
✓ wav2vec2 working.


## 🧠 CELL 5 — Whisper Fine-Tuning using LoRA

This section fine-tunes the Whisper-small model using Low-Rank Adaptation (LoRA). LoRA enables parameter-efficient fine-tuning with reduced computational cost while improving transcription performance for dysarthric speech.

In [ ]:
# ================================================================
# CELL 5 — Whisper + LoRA fine-tuning
# ================================================================

import torch
import librosa
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import get_peft_model, LoraConfig, TaskType
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset as TorchDataset

print("Loading Whisper small...")
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
whisper_base      = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)
whisper_model = get_peft_model(whisper_base, lora_config).to(DEVICE)
whisper_model.print_trainable_parameters()


class WhisperDataset(TorchDataset):
    def __init__(self, samples):
        self.items = []
        for s in samples:
            if not s["text"]:
                continue
            try:
                audio, _ = librosa.load(s["augmented_path"], sr=16000)
                feats    = whisper_processor(
                    audio, sampling_rate=16000, return_tensors="pt"
                ).input_features[0]
                labels   = whisper_processor.tokenizer(
                    s["text"], return_tensors="pt"
                ).input_ids[0]
                self.items.append((feats, labels))
            except Exception as e:
                print(f"  Skip: {e}")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def collate_fn(batch):
    feats  = torch.stack([b[0] for b in batch])
    maxlen = max(b[1].shape[0] for b in batch)
    labels = torch.full((len(batch), maxlen), -100, dtype=torch.long)
    for i, (_, l) in enumerate(batch):
        labels[i, :l.shape[0]] = l
    return feats, labels


ds     = WhisperDataset(augmented_samples)
loader = DataLoader(ds, batch_size=2, shuffle=True, collate_fn=collate_fn)
opt    = AdamW(whisper_model.parameters(), lr=1e-4)

print(f"\nFine-tuning on {len(ds)} samples for 3 epochs...")
whisper_model.train()

# Get the underlying base model to avoid PEFT's broken forward() injection
base_model = whisper_model.base_model.model

for epoch in range(3):
    total_loss = 0
    for feats, labels in loader:
        feats, labels = feats.to(DEVICE), labels.to(DEVICE)

        decoder_input_ids = labels.clone()
        decoder_input_ids[decoder_input_ids == -100] = whisper_processor.tokenizer.pad_token_id

        out = base_model(
            input_features=feats,
            decoder_input_ids=decoder_input_ids,
            labels=labels
        )
        loss = out.loss
        loss.backward()
        opt.step()
        opt.zero_grad()
        total_loss += loss.item()
    print(f"  Epoch {epoch+1}/3 — loss: {total_loss/len(loader):.4f}")

# Still save via the PEFT wrapper so LoRA weights are saved correctly
whisper_model.save_pretrained("./whisper-lora-finetuned")
print("✓ Model saved to ./whisper-lora-finetuned")

def transcribe_whisper(audio_path):
    whisper_model.eval()
    audio, _ = librosa.load(audio_path, sr=16000)
    inp = whisper_processor(
        audio, sampling_rate=16000, return_tensors="pt"
    )
    with torch.no_grad():
        ids = whisper_model.generate(
            input_features=inp["input_features"].to(DEVICE)
        )
    return whisper_processor.batch_decode(ids, skip_special_tokens=True)[0].lower()


# Quick test
result = transcribe_whisper(augmented_samples[0]["original_path"])
print(f"\nTest:")
print(f"  Reference : {augmented_samples[0]['text'].lower()}")
print(f"  Predicted : {result}")
print("✓ Both ASR models ready.")

Loading Whisper small...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


trainable params: 884,736 || all params: 242,619,648 || trainable%: 0.3647

Fine-tuning on 10 samples for 3 epochs...
  Epoch 1/3 — loss: 8.7943
  Epoch 2/3 — loss: 7.0099
  Epoch 3/3 — loss: 5.7500


Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.


✓ Model saved to ./whisper-lora-finetuned

Test:
  Reference : mister quilter is the apostle of the middle classes and we are glad to welcome his gospel
  Predicted :  mr. quilter is the apostle of the middle classes, and we are glad to welcome his gospel.
✓ Both ASR models ready.


## 🔍 CELL 6 — Loading NLP Models

This section loads pretrained NLP models for grammar correction, intent classification, named entity recognition, and sentiment analysis. These models are used for downstream language understanding tasks after ASR transcription.

In [ ]:
# ================================================================
# CELL 6 — Load NLP models
# Grammar | Intent | NER | Sentiment
# ================================================================

from transformers import pipeline
import spacy
import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Loading NLP models...\n")

# 1. Grammar Correction (~240 MB)
grammar_pipe = pipeline(
    "text2text-generation",
    model="prithivida/grammar_error_correcter_v1",
    device=0 if DEVICE == "cuda" else -1
)
print("✓ Grammar ready")

# 2. Intent Classification (~90 MB)
intent_pipe = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-MiniLM2-L6-H768",
    device=0 if DEVICE == "cuda" else -1
)
INTENT_LABELS = [
    "request help", "ask question", "give information",
    "express emotion", "greeting", "command"
]
print("✓ Intent ready")

# 3. NER — spaCy (~12 MB)
nlp_ner = spacy.load("en_core_web_sm")
print("✓ NER ready")

# 4. Sentiment (~270 MB)
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if DEVICE == "cuda" else -1
)
print("✓ Sentiment ready")
print("\n✓ All NLP models loaded (~612 MB total).")

Loading NLP models...

✓ Grammar ready
✓ Intent ready
✓ NER ready
✓ Sentiment ready

✓ All NLP models loaded (~612 MB total).


## 📝 CELL 7 — NLP Processing Pipeline

This section defines the complete NLP processing workflow. The generated transcript passes through grammar correction, intent detection, named entity recognition, and sentiment analysis to produce structured linguistic output.

In [ ]:
# ================================================================
# CELL 7 — NLP pipeline function + test
# ================================================================

def run_nlp_pipeline(raw_text):
    """
    4-stage NLP: grammar → intent → NER → sentiment
    """
    if not raw_text or len(raw_text.strip()) < 3:
        return {"error": "Input too short"}

    output = {"raw_transcript": raw_text}

    # 1. Grammar
    try:
        res       = grammar_pipe(
            f"gec: {raw_text}", max_length=128, num_beams=2, early_stopping=True
        )
        corrected = res[0]["generated_text"].strip() or raw_text
        output["corrected_text"] = corrected
    except Exception as e:
        print(f"  [Grammar] {e}")
        corrected = raw_text
        output["corrected_text"] = raw_text

    # 2. Intent
    try:
        ir = intent_pipe(corrected, candidate_labels=INTENT_LABELS)
        output["intent"] = {
            "label": ir["labels"][0],
            "score": round(ir["scores"][0], 3),
            "all_scores": {l: round(s, 3) for l, s in zip(ir["labels"], ir["scores"])}
        }
    except Exception as e:
        output["intent"] = {"error": str(e)}

    # 3. NER
    try:
        doc = nlp_ner(corrected)
        output["entities"] = [
            {
                "text":        ent.text,
                "label":       ent.label_,
                "description": spacy.explain(ent.label_) or "—"
            }
            for ent in doc.ents
        ]
    except Exception as e:
        print(f"  [NER] {e}")
        output["entities"] = []

    # 4. Sentiment
    try:
        sr = sentiment_pipe(corrected[:512])[0]
        label_map = {
            "positive": "Positive 😊",
            "negative": "Negative 😔",
            "neutral":  "Neutral 😐"
        }
        output["sentiment"] = {
            "label": label_map.get(sr["label"].lower(), sr["label"]),
            "score": round(sr["score"], 3)
        }
    except Exception as e:
        output["sentiment"] = {"error": str(e)}

    return output


# Quick tests
tests = [
    "i want go hospital my head is hurting bad",
    "please call doctor now i cannot breathe good",
    "thank you helping me today i feel much better"
]
print("NLP Tests")
print("=" * 60)
for sent in tests:
    r = run_nlp_pipeline(sent)
    print(f"\n  Input     : {r['raw_transcript']}")
    print(f"  Corrected : {r.get('corrected_text','—')}")
    intent = r.get("intent", {})
    print(f"  Intent    : {intent.get('label','—')} ({intent.get('score','—')})")
    entities = r.get("entities", [])
    print(f"  Entities  : {[e['text'] for e in entities] if entities else 'None'}")
    s = r.get("sentiment", {})
    print(f"  Sentiment : {s.get('label','—')} ({s.get('score','—')})")
    print("-" * 60)
print("\n✓ NLP pipeline ready.")

NLP Tests

  Input     : i want go hospital my head is hurting bad
  Corrected : i want to go to hospital my head is hurting bad.
  Intent    : express emotion (0.74)
  Entities  : None
  Sentiment : Negative 😔 (1.0)
------------------------------------------------------------

  Input     : please call doctor now i cannot breathe good
  Corrected : please call doctor now i cannot breathe good.
  Intent    : request help (0.256)
  Entities  : None
  Sentiment : Negative 😔 (0.998)
------------------------------------------------------------

  Input     : thank you helping me today i feel much better
  Corrected : thank you for helping me today i feel much better.
  Intent    : express emotion (0.676)
  Entities  : ['today']
  Sentiment : Positive 😊 (1.0)
------------------------------------------------------------

✓ NLP pipeline ready.


## 📊 CELL 8 — Evaluation Metrics

This section evaluates ASR and NLP performance using Word Error Rate (WER), Character Error Rate (CER), sentiment classification accuracy, and macro F1-score. Baseline and fine-tuned model performances are compared.

In [ ]:
# ================================================================
# CELL 8 — Evaluation: WER + CER + Sentiment F1
# ================================================================

from jiwer import wer, cer
from sklearn.metrics import f1_score, accuracy_score, classification_report
import pandas as pd

print("Running evaluation...\n")

# ── ASR Evaluation ───────────────────────────────────────────
def evaluate_asr(samples, transcribe_fn, model_name):
    refs, hyps = [], []
    for s in samples:
        if not s["text"]:
            continue
        try:
            ref = s["text"].lower().strip()
            hyp = transcribe_fn(s["original_path"]).strip()
            refs.append(ref)
            hyps.append(hyp)
        except Exception as e:
            print(f"  Skipped: {e}")

    if not refs:
        print(f"{model_name}: No samples evaluated.")
        return None

    word_error = wer(refs, hyps)
    char_error = cer(refs, hyps)

    print(f"{'='*50}")
    print(f"Model   : {model_name}")
    print(f"Samples : {len(refs)}")
    print(f"WER     : {word_error:.3f}  ({word_error*100:.1f}%)")
    print(f"CER     : {char_error:.3f}  ({char_error*100:.1f}%)")
    print("\nSample predictions:")
    for ref, hyp in zip(refs[:3], hyps[:3]):
        print(f"  REF: {ref}")
        print(f"  HYP: {hyp}\n")

    return {
        "Model":   model_name,
        "Samples": len(refs),
        "WER":     round(word_error, 3),
        "CER":     round(char_error, 3)
    }

r_w2v     = evaluate_asr(augmented_samples, transcribe_wav2vec2, "wav2vec2 (baseline)")
r_whisper = evaluate_asr(augmented_samples, transcribe_whisper,  "Whisper + LoRA (fine-tuned)")

# ── Sentiment Evaluation ─────────────────────────────────────
sentiment_test = [
    ("I feel great today",                    "positive"),
    ("my head hurts and I am very tired",     "negative"),
    ("please call the doctor now",            "negative"),
    ("I want to go home",                     "neutral"),
    ("thank you so much for helping me",      "positive"),
    ("I do not understand what is happening", "negative"),
    ("the weather is okay today",             "neutral"),
    ("I am so happy to see you again",        "positive"),
]

true_labels, pred_labels = [], []
for text, true_label in sentiment_test:
    result   = run_nlp_pipeline(text)
    raw_pred = result["sentiment"].get("label", "").lower()
    if   "positive" in raw_pred: pred = "positive"
    elif "negative" in raw_pred: pred = "negative"
    else:                        pred = "neutral"
    true_labels.append(true_label)
    pred_labels.append(pred)

print("=" * 50)
print("Sentiment Evaluation")
print("=" * 50)
print(classification_report(true_labels, pred_labels, zero_division=0))
sent_f1  = f1_score(true_labels, pred_labels, average="macro", zero_division=0)
sent_acc = accuracy_score(true_labels, pred_labels)
print(f"F1 (macro) : {sent_f1:.3f}")
print(f"Accuracy   : {sent_acc:.3f}")

# ── Summary Table ────────────────────────────────────────────
rows = []
if r_w2v:     rows.append(r_w2v)
if r_whisper: rows.append(r_whisper)
rows.append({
    "Model":   "Sentiment (DistilBERT)",
    "Samples": len(sentiment_test),
    "WER":     "—",
    "CER":     f"F1={sent_f1:.3f} / Acc={sent_acc:.3f}"
})

print("\n" + "=" * 50)
print("FINAL RESULTS SUMMARY")
print("=" * 50)
print(pd.DataFrame(rows).to_string(index=False))

Running evaluation...

Model   : wav2vec2 (baseline)
Samples : 10
WER     : 0.059  (5.9%)
CER     : 0.022  (2.2%)

Sample predictions:
  REF: mister quilter is the apostle of the middle classes and we are glad to welcome his gospel
  HYP: mister quilter is the apostle of the middle classes and we are glad to welcome his gospel

  REF: nor is mister quilter's manner less interesting than his matter
  HYP: nor is mister quilter's manner less interesting than his matter

  REF: he tells us that at this festive season of the year with christmas and roast beef looming before us similes drawn from eating and its results occur most readily to the mind
  HYP: he tells us that at this festive season of the year with christmanus and roast beef looming before us similes drawn from eating and its results occur most readily to the mind

Model   : Whisper + LoRA (fine-tuned)
Samples : 10
WER     : 0.189  (18.9%)
CER     : 0.063  (6.3%)

Sample predictions:
  REF: mister quilter is the apostle of the

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Sentiment Evaluation
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00         3
     neutral       0.00      0.00      0.00         2
    positive       0.60      1.00      0.75         3

    accuracy                           0.75         8
   macro avg       0.53      0.67      0.58         8
weighted avg       0.60      0.75      0.66         8

F1 (macro) : 0.583
Accuracy   : 0.750

FINAL RESULTS SUMMARY
                      Model  Samples    WER                  CER
        wav2vec2 (baseline)       10  0.059                0.022
Whisper + LoRA (fine-tuned)       10  0.189                0.063
     Sentiment (DistilBERT)        8      — F1=0.583 / Acc=0.750


## 🌐 CELL 9 — Interactive Gradio Deployment

This section creates an interactive Gradio-based web application for real-time dysarthric speech recognition and NLP analysis. Users can upload audio or record speech directly through the interface.

In [ ]:
# ================================================================
# CELL 9 — Gradio Demo
# ================================================================

import gradio as gr
import numpy as np
import soundfile as sf
import librosa
import json

def full_pipeline(audio, model_choice):

    # No audio
    if audio is None:
        return "No audio received.", "", "", "", "", "", None

    # Guard against unexpected types
    if not isinstance(audio, (tuple, list)) or len(audio) != 2:
        return "Unexpected audio format.", "", "", "", "", "", None

    # ============================================================
    # Audio Preprocessing
    # ============================================================

    try:
        sr, audio_array = audio

        if audio_array is None or len(audio_array) == 0:
            return "Empty audio received.", "", "", "", "", "", None

        audio_float = audio_array.astype(np.float32)

        # Stereo → Mono
        if audio_float.ndim == 2:
            audio_float = audio_float.mean(axis=1)

        # Normalize if needed
        if np.abs(audio_float).max() > 1.0:
            audio_float /= 32768.0

        # Resample to 16kHz
        if sr != 16000:
            audio_float = librosa.resample(
                audio_float,
                orig_sr=sr,
                target_sr=16000
            )

        # Save processed audio
        temp_path = "/tmp/gradio_input.wav"

        sf.write(temp_path, audio_float, 16000)

    except Exception as e:
        return f"Audio error: {e}", "", "", "", "", "", None

    # ============================================================
    # ASR
    # ============================================================

    try:

        if model_choice == "Whisper (fine-tuned with LoRA)":
            raw_transcript = transcribe_whisper(temp_path)

        else:
            raw_transcript = transcribe_wav2vec2(temp_path)

    except Exception as e:
        return f"ASR error: {e}", "", "", "", "", "", None

    # ============================================================
    # NLP Pipeline
    # ============================================================

    try:
        nlp_out = run_nlp_pipeline(raw_transcript)

    except Exception as e:
        return raw_transcript, f"NLP error: {e}", "", "", "", "", temp_path

    corrected = nlp_out.get("corrected_text", raw_transcript)

    # Intent
    intent_info = nlp_out.get("intent", {})

    intent_str = (
        f"{intent_info.get('label', 'N/A')} "
        f"(confidence: {intent_info.get('score', 'N/A')})"
    )

    # Entities
    entities = nlp_out.get("entities", [])

    entity_str = (
        "\n".join([
            f"• {e['text']} → {e['label']} ({e['description']})"
            for e in entities
        ])
        if entities else "No named entities detected."
    )

    # Sentiment
    sentiment = nlp_out.get("sentiment", {})

    sent_str = (
        f"{sentiment.get('label', 'N/A')} "
        f"(score: {sentiment.get('score', 'N/A')})"
    )

    # Full JSON
    full_json = json.dumps(
        nlp_out,
        indent=2,
        ensure_ascii=False
    )

    return (
        raw_transcript,
        corrected,
        intent_str,
        entity_str,
        sent_str,
        full_json,
        temp_path
    )


# ================================================================
# Gradio UI
# ================================================================

with gr.Blocks(
    title="Dysarthric Speech Recognition",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown("""
    # 🎙️ AI-Based Dysarthric Speech Recognition System

    **Speak into the mic or upload a .wav file → get full NLP analysis**

    *Pipeline: Audio → ASR → Grammar Fix → Intent → NER → Sentiment*
    """)

    with gr.Row():

        # ========================================================
        # LEFT PANEL
        # ========================================================

        with gr.Column(scale=1):

            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="numpy",
                label="🎤 Audio Input"
            )

            model_radio = gr.Radio(
                choices=[
                    "Whisper (fine-tuned with LoRA)",
                    "wav2vec2 (baseline)"
                ],
                value="Whisper (fine-tuned with LoRA)",
                label="ASR Model"
            )

            run_btn = gr.Button(
                "▶ Run Pipeline",
                variant="primary",
                size="lg"
            )

        # ========================================================
        # RIGHT PANEL
        # ========================================================

        with gr.Column(scale=2):

            out_raw = gr.Textbox(
                label="📝 Raw ASR Transcript",
                lines=2
            )

            out_corrected = gr.Textbox(
                label="✏️ Grammar-Corrected Text",
                lines=2
            )

            out_intent = gr.Textbox(
                label="🎯 Intent",
                lines=1
            )

            out_entities = gr.Textbox(
                label="🔍 Named Entities",
                lines=3
            )

            out_sentiment = gr.Textbox(
                label="💬 Sentiment",
                lines=1
            )

            out_json = gr.Code(
                label="📊 Full JSON Output",
                language="json"
            )

            out_audio = gr.Audio(
                label="🔊 Input Audio (Normalized)",
                type="filepath"
            )

    # ============================================================
    # Button Action
    # ============================================================

    run_btn.click(
        fn=full_pipeline,
        inputs=[audio_input, model_radio],
        outputs=[
            out_raw,
            out_corrected,
            out_intent,
            out_entities,
            out_sentiment,
            out_json,
            out_audio
        ]
    )

print("Launching Gradio app...")

demo.launch(share=True)

Launching Gradio app...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://cb441acdf2fef462d4.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
